# Hollywood by the Numbers: Clustering Analysis
- Alex Arce: aarce
- Lunden Mandigo: lundenm
- Tyrone Pettygrue: tpetty

## Preprocessing Data

In [6]:
import pandas as pd
from sklearn.pipeline import Pipeline


In [92]:
movies = pd.read_csv('movies.csv')
# drop a column
movies.drop(columns=['originalTitle'], inplace=True)
movies.drop(columns=['main_genre'], inplace=True)
movies.drop(columns=['rating_category'], inplace=True)

# combine two columns into one
movies['title'] = movies['title'] + ' (' + movies['year'].astype(str) + ')'

movies.drop_duplicates(subset=['title'], inplace=True)


# make a column the index
movies.set_index('title', inplace=True)

# Split a column of strings into lists
movies['genres'] = movies['genres'].str.split(', ')
movies['productionCountries'] = movies['productionCountries'].str.split(', ')

# Drop duplicate rows based on the title and year column
# movies.drop_duplicates(subset=['title', 'year'], inplace=True)

# replace NaN values with empty lists
movies['genres'].fillna(value={}, inplace=True)
movies['productionCountries'].fillna(value={}, inplace=True)

# drop all values that are not list objects
movies['productionCountries'] = movies['productionCountries'].apply(lambda x: x if isinstance(x, list) else ([x] if pd.notna(x) else []))



/var/folders/bg/jwt2p6k50v7cvkml3qmd1kz40000gn/T/ipykernel_8415/152500526.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movies['genres'].fillna(value={}, inplace=True)
/var/folders/bg/jwt2p6k50v7cvkml3qmd1kz40000gn/T/ipykernel_8415/152500526.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always 

In [59]:
movies.head()


,isAdult,runtimeMinutes,genres,IMDBavgRating,numVotes,rank,worldwideGross,domesticGross,domestic%,foreignGross,foreign%,year,originalLang,productionCountries
title,,,,,,,,,,,,,,
Kate & Leopold (2001),0,118,"[Comedy,Fantasy,Romance]",6.4,91304,56,76019048.0,47121859.0,62.0,28897189.0,38.0,2001,en,[United States of America]
The Sorcerer's Apprentice (2010),0,86,"[Adventure,Family,Fantasy]",4.2,757,34,215283742.0,63150991.0,29.3,152132751.0,70.7,2010,en,[United States of America]
Fantastic Four (2005),0,106,"[Action,Adventure,Fantasy]",5.7,353227,11,333535934.0,154696080.0,46.4,178839854.0,53.6,2005,en,"[Germany, United States of America]"
Fantastic Four (2015),0,106,"[Action,Adventure,Fantasy]",5.7,353227,44,167882881.0,56117548.0,33.4,111765333.0,66.6,2015,en,"[United Kingdom, Germany, United States of Ame..."
Frida (2002),0,123,"[Biography,Drama,Romance]",7.3,97408,78,56298474.0,25885000.0,46.0,30413474.0,54.0,2002,en,"[Canada, Mexico, United States of America]"


In [47]:
movies.productionCountries.value_counts().sort_values(ascending=True).head(10)

productionCountries
[Slovakia, Spain, United Kingdom, United States of America]             1
[Dominican Republic, United Kingdom, United States of America]          1
[Brazil, China, United States of America]                               1
[India, United States of America, United Arab Emirates]                 1
[Australia, France, United Kingdom]                                     1
[France, Germany, Ireland, United Kingdom, United States of America]    1
[Spain, United Kingdom]                                                 1
[Belgium, France, United Kingdom]                                       1
[United Kingdom, Poland]                                                1
[Canada, Denmark, United States of America]                             1
Name: count, dtype: int64

In [101]:
# create a pipeline that scales numerical columns using standard scaler and onehot encodes categorical columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

mlb = MultiLabelBinarizer()
encoded_genres = mlb.fit_transform(movies['genres'])
encoded_genres_df = pd.DataFrame(encoded_genres, columns=mlb.classes_)
encoded_genres_df.fillna(0, inplace=True)
encoded_production_countries = mlb.fit_transform(movies['productionCountries'])
encoded_production_countries_df = pd.DataFrame(encoded_production_countries, columns=mlb.classes_)
encoded_production_countries_df.fillna(0, inplace=True)



# # concat the encoded columns with the original dataframe
processed_movies = pd.concat([movies, encoded_genres_df, encoded_production_countries_df], axis=1)
# # drop the original columns
processed_movies.drop(columns=['genres'], inplace=True)
processed_movies.drop(columns=['productionCountries'], inplace=True)
processed_movies.fillna(0, inplace=True)
processed_movies.drop(columns=['originalLang'], inplace=True)

# onehot encode the categorical columns
categorical_cols = ['originalLang']
numerical_cols = ['runtimeMinutes', 'IMDBavgRating', 'numVotes', 'rank', 'worldwideGross', 'domesticGross', 'domestic%', 'foreignGross', 'foreign%', "year"]




In [106]:
Yid = processed_movies.index

In [102]:
num_pipeline = Pipeline([
    ('impute',SimpleImputer(strategy='median')), 
    ('scale',StandardScaler())
    ])

preprocessing_pipeline = ColumnTransformer([
    ('num', num_pipeline, numerical_cols)])

In [103]:
# apply the pipeline to my data
scaled_X = preprocessing_pipeline.fit_transform(processed_movies)

In [116]:
# apply PCA to the scaled data
model = PCA(n_components=7)
X_pca = model.fit_transform(scaled_X)
# create a dataframe from the PCA data
pca_df = pd.DataFrame(X_pca,index=Yid, columns=[f'PC{i}' for i in range(1, 8)])
pca_df.head()
model.explained_variance_ratio_


array([0.55933881, 0.26002528, 0.09674299, 0.04601094, 0.01807539,
       0.01108959, 0.00583876])

In [117]:
pca_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7
Kate & Leopold (2001),1.864249,0.535167,1.323470,-0.164700,-0.695228,0.089683,0.101921
The Sorcerer's Apprentice (2010),1.767660,-0.491555,-0.530923,-0.987899,-0.594355,-0.008567,-0.220475
Fantastic Four (2005),3.311795,-2.336796,0.485482,0.383208,-0.328572,0.479987,-0.072973
Fantastic Four (2015),2.489338,-0.653445,0.074520,1.192667,-0.387155,-0.165385,-0.140464
Frida (2002),1.955079,1.009279,0.638201,0.110975,-0.587277,-0.012934,0.065140


In [118]:
# write the data to a csv
pca_df.to_csv('preprocessed_movie_data.csv')
